# Uniform-cuboid FMM versus exact dense cuboid direct

This notebook uses the validated exact `DenseDirectPlan` cube-to-point field as its all-to-all baseline. Targets coincide with cube centres, so every finite self field is included. It compares two `UNIFORM_CUBOID -> POINT` FMM variants: exact cuboid tensors in P2P with either finite-cuboid P2M or ordinary point-dipole P2M. The comparison therefore isolates the accuracy and runtime effect of representing finite source extent in the upward pass.


In [1]:
import time
import cdfmm
import matplotlib.pyplot as plt
import numpy as np

SIDE = 10.0e-9
SPACING = 3.0 * SIDE
SHAPE = (15, 15, 15)
ORDERS = [6, 8]
TREE_DEPTHS = [2, 3, 4, 5]
P2M_CASES = [
    ("cube P2M + cube P2P", True),
    ("dipole P2M + cube P2P", False),
]
REPEATS = 1

indices = np.indices(SHAPE, dtype=float).reshape(3, -1).T
centres = (indices - (np.asarray(SHAPE) - 1) / 2) * SPACING
phase = np.arange(len(centres), dtype=float)
magnetisations = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * phase),
    -3.0e5 + 2.0e5 * np.cos(0.11 * phase),
    4.0e5 * np.sin(0.07 * phase + 0.3),
))
cube = cdfmm.CuboidSize(SIDE, SIDE, SIDE)
moments = SIDE**3 * magnetisations


## Exact reusable reference

Construction and repeated evaluation are timed separately. No MagTense dependency is involved.


In [2]:
start = time.perf_counter()
direct = cdfmm.DenseDirectPlan(
    source_positions=centres,
    target_positions=centres,
    source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
    target_geometry=cdfmm.TargetGeometry.POINT,
    source_sizes=[cube],
)
direct_initialisation_s = time.perf_counter() - start
direct.evaluate(moments, backend=cdfmm.DenseDirectBackend.PORTABLE)
samples = []
for _ in range(REPEATS):
    start = time.perf_counter()
    H_direct = direct.evaluate(moments, backend=cdfmm.DenseDirectBackend.PORTABLE)
    samples.append(time.perf_counter() - start)
direct_evaluation_s = np.median(samples)
print(f"Dense construction: {direct_initialisation_s:.6f} s")
print(f"Dense evaluation:   {direct_evaluation_s:.6f} s")


Dense construction: 7.800497 s
Dense evaluation:   0.058719 s


## Expansion-order convergence and timing

The meaningful pointwise denominator excludes fields smaller than $10^{-12}$ of the maximum reference magnitude. Order five is included because cube symmetry makes it the first degree carrying the physical finite-size correction to the external point-dipole field. Every expansion order is evaluated at tree depths 2, 3, and 4 for both P2M formulations. In both cases, list1 P2P remains the exact finite-cube interaction, including self-fields.


In [ ]:
rows = []
reference_norms = np.linalg.norm(H_direct, axis=1)
meaningful = reference_norms > 1.0e-12 * reference_norms.max()
for depth in TREE_DEPTHS:
    for p2m_case, use_cuboid_p2m in P2M_CASES:
        for order in ORDERS:
            options = cdfmm.UniformFmmOptions()
            options.expansion_basis = cdfmm.ExpansionBasis.CARTESIAN
            options.expansion_order = order
            options.tree.max_level = depth
            options.source_geometry = cdfmm.SourceGeometry.UNIFORM_CUBOID
            options.source_sizes = [cube]
            options.use_cuboid_p2m = use_cuboid_p2m
            options.backend = cdfmm.ExecutionBackend.CPU_STATIC
            options.fixed_target_source_indices = list(range(len(centres)))
            start = time.perf_counter()
            fmm = cdfmm.UniformFmm(centres, centres, options)
            initialisation_s = time.perf_counter() - start
            fmm.evaluate(moments)
            samples = []
            for _ in range(REPEATS):
                start = time.perf_counter()
                H_fmm = fmm.evaluate(moments)["H"]
                samples.append(time.perf_counter() - start)
            difference = H_fmm - H_direct
            absolute = np.linalg.norm(difference, axis=1)
            rows.append(dict(
                p2m_case=p2m_case,
                use_cuboid_p2m=use_cuboid_p2m,
                depth=depth,
                order=order,
                initialisation_s=initialisation_s,
                evaluation_s=float(np.median(samples)),
                relative_l2=float(
                    np.linalg.norm(difference) / np.linalg.norm(H_direct)
                ),
                maximum_absolute=float(absolute.max()),
                maximum_pointwise_relative=float(
                    np.max(absolute[meaningful] / reference_norms[meaningful])
                ),
            ))

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for depth_index, depth in enumerate(TREE_DEPTHS):
    colour = f"C{depth_index}"
    case_rows = {}
    for p2m_case, use_cuboid_p2m in P2M_CASES:
        selected = [
            row for row in rows
            if row["depth"] == depth and row["p2m_case"] == p2m_case
        ]
        case_rows[use_cuboid_p2m] = selected
        orders = [row["order"] for row in selected]
        errors = [row["relative_l2"] for row in selected]
        times = [row["evaluation_s"] for row in selected]
        linestyle = "-" if use_cuboid_p2m else "--"
        label = f"depth {depth}: {p2m_case}"
        axes[0].semilogy(
            orders, errors, marker="o", linestyle=linestyle,
            color=colour, label=label,
        )
        axes[1].semilogy(
            orders, times, marker="o", linestyle=linestyle,
            color=colour, label=label,
        )
    cube_errors = np.asarray([
        row["relative_l2"] for row in case_rows[True]
    ])
    dipole_errors = np.asarray([
        row["relative_l2"] for row in case_rows[False]
    ])
    improvement = dipole_errors / np.maximum(
        cube_errors, np.finfo(float).tiny
    )
    axes[2].semilogy(
        ORDERS, improvement, "o-", color=colour, label=f"depth {depth}"
    )
axes[0].set(xlabel="Expansion order p", ylabel="Relative L2 error",
            title="Accuracy against exact cube direct")
axes[1].set(xlabel="Expansion order p", ylabel="Evaluation time [s]",
            title="Repeated FMM evaluation")
axes[2].axhline(1.0, color="black", linewidth=0.8)
axes[2].set(
    xlabel="Expansion order p",
    ylabel="dipole-P2M error / cube-P2M error",
    title="Cube-P2M accuracy factor (>1 is better)",
)
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()


## Scope

This validates axis-aligned uniformly magnetised cuboid sources and point targets on the CPU static backend. Both variants use exact cube-to-point P2P; only the P2M source representation changes. It does not introduce point-to-cuboid, cuboid-averaged targets, rotated cuboids, or target-volume L2P.
